# Анализ источников финансирования ЛУКОЙЛ

Этот notebook запускает тот же воспроизводимый pipeline, что и CLI проекта. Он отделяет точную реконструкцию Excel от модели с единым определением капитала.

## Входы и ограничения

В репозитории хранится обезличенный JSON-снимок листов Excel с формулами и сохранёнными значениями. Денежные поля исходника заданы в тыс. ₽. Аналитическая модель использует бухгалтерский собственный капитал и процентный долг. В источнике три годовых наблюдения, поэтому notebook показывает анализ и чувствительность, а не прогноз ML.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src/financing_analysis").exists():
    PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from financing_analysis.pipeline import run_analysis  # noqa: E402

result = run_analysis(PROJECT_ROOT)
frames = result["frames"]
summary = result["verification_summary"]
summary

## Проверка переноса формул

Рекалькулятор работает с ограниченным Excel-синтаксисом исходной книги и не передаёт формулы в `eval`. Ошибки деления на ноль остаются явными.

In [ ]:
import pandas as pd

pd.DataFrame(result["verification"]).query("status != 'match'")[
    ["sheet", "cell", "formula", "cached", "calculated", "status"]
]

## Исторические показатели

Таблица ниже объединяет извлечённые входы, средние безрисковые ставки, CAPM, стоимость долга после налога и WACC.

In [ ]:
annual_columns = [
    "year",
    "revenue_k_rub",
    "net_profit_k_rub",
    "net_margin",
    "book_equity_k_rub",
    "interest_bearing_debt_k_rub",
    "canonical_debt_share",
    "risk_free_rate",
    "cost_of_equity",
    "cost_of_debt_after_tax",
    "wacc",
    "roe",
]
frames["annual"][annual_columns].assign(
    net_margin=lambda frame: frame["net_margin"] * 100,
    canonical_debt_share=lambda frame: frame["canonical_debt_share"] * 100,
    risk_free_rate=lambda frame: frame["risk_free_rate"] * 100,
    cost_of_equity=lambda frame: frame["cost_of_equity"] * 100,
    cost_of_debt_after_tax=lambda frame: frame["cost_of_debt_after_tax"] * 100,
    wacc=lambda frame: frame["wacc"] * 100,
    roe=lambda frame: frame["roe"] * 100,
).round(2)

## Сценарии и визуализации

Доля долга 100% помечена как недоступная: собственный капитал равен нулю, поэтому D/E, Hamada и ROE не определены. Минимум WACC в детерминированной сетке не является рекомендацией.

In [ ]:
frames["scenarios"][
    [
        "debt_share",
        "equity_share",
        "levered_beta",
        "cost_of_equity",
        "cost_of_debt_after_tax",
        "wacc",
        "roe",
        "status",
    ]
].assign(
    debt_share=lambda frame: frame["debt_share"] * 100,
    equity_share=lambda frame: frame["equity_share"] * 100,
    cost_of_equity=lambda frame: frame["cost_of_equity"] * 100,
    cost_of_debt_after_tax=lambda frame: frame["cost_of_debt_after_tax"] * 100,
    wacc=lambda frame: frame["wacc"] * 100,
    roe=lambda frame: frame["roe"] * 100,
).round(2)

In [ ]:
from IPython.display import Image, display

for path in result["figures"]:
    display(Image(filename=str(path)))

## Что читать дальше

`reports/analysis_report.md` содержит основные выводы. `docs/source-audit.md` перечисляет конкретные координаты ошибок Excel, а `docs/methodology.md` формализует единицы, формулы и ограничения.